In [ ]:
# 0) Setup
import re
import numpy as np
import pandas as pd
from pathlib import Path

PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)
RAW_CSV = "data/raw/all_measurements.csv"  # attached file
OUT_X = PROC_DIR / "ibc_processed.csv"
OUT_Y = Path("data/labels_filtered.csv")

# 1) Load and sanitize
df = pd.read_csv(RAW_CSV)
assert "subject_id" in df.columns, "subject_id column missing"
print("Raw shape:", df.shape)

# Keep numeric cols; drop rows with any NaN/inf in gain columns later
num_df = df.copy()

# 2) Detect rx_gain families and parse frequency in Hz
def parse_gain_columns(columns, family_prefix):
    pat = re.compile(rf"^{family_prefix}_f_(\d+)$")
    pairs = []
    for c in columns:
        m = pat.match(c)
        if m:
            f = float(m.group(1))
            pairs.append((f, c))
    pairs.sort(key=lambda x: x[0])
    freqs = np.array([p[0] for p in pairs], dtype=float)
    cols = [p[1] for p in pairs]
    return freqs, cols

freqs_50, cols_50 = parse_gain_columns(num_df.columns, "rx_gain_50")
freqs_1M, cols_1M = parse_gain_columns(num_df.columns, "rx_gain_1M")
print(f"Found 50Ω: {len(cols_50)} freqs, 1MΩ: {len(cols_1M)} freqs")

# 3) Choose family with lower zero/missing ratio across rows
def family_quality(subdf, cols):
    if not cols:
        return np.inf
    vals = subdf[cols].replace([np.inf, -np.inf], np.nan)
    zero_or_nan = ((vals == 0) | (vals.isna())).sum(axis=1).mean()
    return float(zero_or_nan)

q_50 = family_quality(num_df, cols_50)
q_1M = family_quality(num_df, cols_1M)
if q_50 < q_1M:
    freqs, cols, family = freqs_50, cols_50, "rx_gain_50"
else:
    freqs, cols, family = freqs_1M, cols_1M, "rx_gain_1M"
print(f"Selected family: {family} with {len(cols)} freqs")

# 4) Drop rows with NaN/inf in selected family; keep aligned labels
vals = num_df[cols].replace([np.inf, -np.inf], np.nan)
mask = ~vals.isna().any(axis=1)
df_clean = num_df.loc[mask].reset_index(drop=True)
vals = vals.loc[mask].reset_index(drop=True)
labels = df_clean["subject_id"].astype(int).values
print("After drop:", vals.shape, "labels:", labels.shape)

# 5) Interpolate to fixed-length spectrum (e.g., 256) over min..max frequency
from numpy.typing import ArrayLike

def resample_spectrum(row_vals: ArrayLike, in_freqs: np.ndarray, out_len: int = 256) -> np.ndarray:
    # In dB already; keep dB then standardize downstream
    fmin, fmax = float(in_freqs.min()), float(in_freqs.max())
    f_out = np.linspace(fmin, fmax, out_len, dtype=float)
    y = np.asarray(row_vals, dtype=float)
    # If any remaining zeros that are true zeros (not NaN), keep as-is; interpolate across finite values
    return np.interp(f_out, in_freqs, y)

# Build X_raw_resampled
X_spec = np.vstack([resample_spectrum(vals.iloc[i].values, freqs, out_len=256)
                    for i in range(len(vals))])
print("Spectrum matrix:", X_spec.shape)

# 6) Save spectrum and labels
pd.DataFrame(X_spec, columns=[f"f{i}" for i in range(X_spec.shape[1])]).to_csv(OUT_X, index=False)
pd.DataFrame({"subject_id": labels}).to_csv(OUT_Y, index=False)
print(f"Saved → {OUT_X}, {OUT_Y}")


In [ ]:
# 7) Derive simple & DWT features from resampled spectrum
import pywt

X_raw = pd.read_csv(OUT_X).values
y = pd.read_csv(OUT_Y)["subject_id"].values

# Simple: band stats on frequency index ranges (예: 8–12.5–15 MHz 근방을 인덱스로 근사)
def band_stats(X, idx_bands):
    feats = []
    for row in X:
        f = []
        for sl in idx_bands:
            seg = row[sl]
            f.extend([float(seg.mean()), float(seg.std())])
        # global slope (simple linear fit)
        x = np.arange(len(row))
        a, b = np.polyfit(x, row, 1)
        f.extend([float(a), float(b)])
        feats.append(f)
    return np.asarray(feats, dtype=float)

# Example bands (adjust to frequency grid; here using indices on 256)
bands = [slice(64, 96), slice(96, 128), slice(128, 160)]
X_simple = band_stats(X_raw, bands)
print("Simple shape:", X_simple.shape)

# DWT(db4, level=2): energy, entropy, mean, std per coeff array
def dwt_stats(X, wavelet="db4", level=2):
    out = []
    for row in X:
        coeffs = pywt.wavedec(row, wavelet, level=level, mode="periodization")
        f = []
        for c in coeffs:
            e = float(np.sum(c**2))
            p = (c**2)/e if e > 0 else np.zeros_like(c)
            ent = float(-np.sum(p*np.log2(p + 1e-12)))
            f.extend([e, ent, float(c.mean()), float(c.std())])
        out.append(f)
    return np.asarray(out, dtype=float)

X_dwt = dwt_stats(X_raw, wavelet="db4", level=2)
print("DWT shape:", X_dwt.shape)

# Optionally save features
pd.DataFrame(X_simple).to_csv(PROC_DIR / "features_simple.csv", index=False)
pd.DataFrame(X_dwt).to_csv(PROC_DIR / "features_dwt_db4_l2.csv", index=False)


In [ ]:
# data_validation.ipynb (single cell-friendly)
# PEP8 + Google Style docstrings

import hashlib
import json
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit


def hash_rows(X: np.ndarray, n_bytes: int = 16) -> np.ndarray:
    """Hash each row deterministically to detect duplicates/leakage.

    Args:
        X: 2D array-like, shape (N, D).
        n_bytes: truncate digest to this many bytes for compactness.

    Returns:
        Array of hex digests per row.
    """
    X = np.asarray(X)
    digests = []
    for i in range(X.shape[0]):
        h = hashlib.blake2b(X[i].tobytes(), digest_size=n_bytes).hexdigest()
        digests.append(h)
    return np.asarray(digests)


def feature_stats(X: np.ndarray) -> Dict[str, float]:
    """Compute global statistics over features.

    Args:
        X: 2D array-like, shape (N, D).

    Returns:
        Dict with summary stats across all elements.
    """
    X = np.asarray(X, dtype=float)
    return {
        "min": float(np.nanmin(X)),
        "p01": float(np.nanpercentile(X, 1)),
        "p25": float(np.nanpercentile(X, 25)),
        "mean": float(np.nanmean(X)),
        "median": float(np.nanpercentile(X, 50)),
        "p75": float(np.nanpercentile(X, 75)),
        "p99": float(np.nanpercentile(X, 99)),
        "max": float(np.nanmax(X)),
        "std": float(np.nanstd(X)),
    }


def count_nan_inf_zero(X: np.ndarray) -> Dict[str, int]:
    """Count NaN, inf, and all-zero rows.

    Args:
        X: 2D array-like.

    Returns:
        Dict with counts of anomalies.
    """
    X = np.asarray(X)
    nan = np.isnan(X).sum()
    inf = np.isinf(X).sum()
    zero_rows = int(np.sum(np.all(X == 0, axis=1)))
    return {"nan": int(nan), "inf": int(inf), "zero_rows": zero_rows}


def constant_near_constant_cols(
    X: np.ndarray, rtol: float = 1e-6, near_ratio: float = 0.99
) -> Tuple[int, int, np.ndarray, np.ndarray]:
    """Detect constant and near-constant columns.

    Args:
        X: 2D array-like.
        rtol: tolerance to consider values equal.
        near_ratio: fraction threshold for near-constant within a single value.

    Returns:
        n_const, n_near, idx_const, idx_near arrays.
    """
    X = np.asarray(X)
    n, d = X.shape
    idx_const = []
    idx_near = []
    for j in range(d):
        col = X[:, j]
        if np.all(np.isclose(col, col[0], rtol=rtol)):
            idx_const.append(j)
        else:
            vals, counts = np.unique(np.round(col, 6), return_counts=True)
            if counts.max() / n >= near_ratio:
                idx_near.append(j)
    return len(idx_const), len(idx_near), np.asarray(idx_const), np.asarray(idx_near)


def high_corr_pairs(X: np.ndarray, thresh: float = 0.995, max_pairs: int = 50):
    """Find highly correlated column pairs.

    Args:
        X: 2D array-like.
        thresh: absolute correlation threshold.
        max_pairs: cap number of reported pairs.

    Returns:
        List of (i, j, corr) sorted by |corr| descending.
    """
    X = np.asarray(X, dtype=float)
    C = np.corrcoef(X, rowvar=False)
    d = C.shape[0]
    pairs = []
    for i in range(d):
        for j in range(i + 1, d):
            c = C[i, j]
            if np.isfinite(c) and abs(c) >= thresh:
                pairs.append((i, j, float(c)))
    pairs.sort(key=lambda t: abs(t[2]), reverse=True)
    return pairs[:max_pairs]


def outlier_ratio_zscore(X: np.ndarray, z: float = 8.0) -> float:
    """Compute fraction of elements with |z| >= threshold.

    Args:
        X: 2D array-like.
        z: z-score threshold.

    Returns:
        Fraction of outliers across all elements.
    """
    X = np.asarray(X, dtype=float)
    mu = np.nanmean(X, axis=0, keepdims=True)
    sd = np.nanstd(X, axis=0, keepdims=True) + 1e-8
    Z = (X - mu) / sd
    frac = float(np.mean(np.abs(Z) >= z))
    return frac


def class_distribution(y: np.ndarray) -> Dict[str, float]:
    """Summarize class counts and imbalance indices.

    Args:
        y: 1D labels.

    Returns:
        Dict with n_classes, min/max/mean counts, gini approx, entropy.
    """
    y = np.asarray(y)
    values, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    gini = 1.0 - float(np.sum(p ** 2))
    ent = float(-np.sum(p * np.log2(np.clip(p, 1e-12, 1.0))))
    return {
        "n_classes": int(len(values)),
        "count_min": int(counts.min()),
        "count_max": int(counts.max()),
        "count_mean": float(counts.mean()),
        "gini": gini,
        "entropy": ent,
    }


def stratified_split_checks(
    X: np.ndarray, y: np.ndarray, test_size: float = 0.2, seed: int = 42
) -> Dict[str, float]:
    """Check stratified split class overlap and leakage via row hashes.

    Args:
        X: features.
        y: labels.
        test_size: validation fraction.
        seed: RNG seed.

    Returns:
        Dict with overlap and leakage flags.
    """
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_idx, va_idx = next(sss.split(X, y))
    Xtr, Xva = X[tr_idx], X[va_idx]
    ytr, yva = y[tr_idx], y[va_idx]
    c_tr, c_va = set(np.unique(ytr)), set(np.unique(yva))
    missing_in_tr = list(sorted(c_va - c_tr))
    h_tr, h_va = hash_rows(Xtr), hash_rows(Xva)
    leakage = bool(len(set(h_tr).intersection(set(h_va))) > 0)
    return {
        "classes_train": float(len(c_tr)),
        "classes_val": float(len(c_va)),
        "val_minus_train_classes": float(len(missing_in_tr)),
        "leakage_detected": float(leakage),
    }


def main():
    # Paths produced by re-extraction pipeline
    proc_x = Path("data/processed/ibc_processed.csv")
    proc_y = Path("data/labels_filtered.csv")
    assert proc_x.exists() and proc_y.exists(), "Processed files not found"

    X = pd.read_csv(proc_x).values
    y = pd.read_csv(proc_y)["subject_id"].astype(int).values

    report = {}
    report["shape_N"] = int(X.shape[0])
    report["shape_D"] = int(X.shape[1])
    report["dtype_numeric"] = bool(np.issubdtype(X.dtype, np.number))
    report["nan_inf_zero"] = count_nan_inf_zero(X)
    report["feature_stats"] = feature_stats(X)
    c_const, c_near, idx_const, idx_near = constant_near_constant_cols(X)
    report["const_cols"] = int(c_const)
    report["near_const_cols"] = int(c_near)
    report["near_const_cols_idx_sample"] = [int(i) for i in idx_near[:10]]
    report["outlier_ratio_z>=8"] = outlier_ratio_zscore(X, z=8.0)
    corr_pairs = high_corr_pairs(X, thresh=0.995, max_pairs=20)
    report["high_corr_pairs_count"] = int(len(corr_pairs))
    report["high_corr_pairs_sample"] = [(int(i), int(j), float(c)) for i, j, c in corr_pairs[:10]]

    # Duplicates
    row_hash = hash_rows(X)
    dup_counts = pd.Series(row_hash).value_counts()
    n_dup_rows = int((dup_counts > 1).sum())
    report["duplicate_rows"] = n_dup_rows

    # Labels
    report["labels_dist"] = class_distribution(y)
    report["labels_type_int"] = bool(np.issubdtype(y.dtype, np.integer))

    # Split checks
    report["split_checks"] = stratified_split_checks(X, y, test_size=0.2, seed=42)

    # Save artifacts
    out_dir = Path("results")
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "data_validation_report.json", "w") as f:
        json.dump(report, f, indent=2)
    # Also export class distribution table
    cls_vals, cls_cnts = np.unique(y, return_counts=True)
    pd.DataFrame({"subject_id": cls_vals, "count": cls_cnts}).to_csv(
        out_dir / "class_distribution.csv", index=False
    )
    print("Validation report saved to results/data_validation_report.json")
    print("Class distribution saved to results/class_distribution.csv")

    # Human-readable summary
    print("--- Summary ---")
    print(f"Shape: N={report['shape_N']}, D={report['shape_D']}")
    print(f"NaN={report['nan_inf_zero']['nan']}, inf={report['nan_inf_zero']['inf']}, zero_rows={report['nan_inf_zero']['zero_rows']}")
    print(f"Const cols={report['const_cols']}, Near-const cols={report['near_const_cols']}")
    print(f"High-corr pairs(>=0.995)={report['high_corr_pairs_count']}")
    print(f"Outlier ratio |z|>=8: {report['outlier_ratio_z>=8']:.6f}")
    print(f"Classes: {int(report['labels_dist']['n_classes'])}, Leakage: {bool(report['split_checks']['leakage_detected'])}")
    if report["split_checks"]["val_minus_train_classes"] > 0:
        print("Warning: some val classes missing in train (check stratification)")

if __name__ == "__main__":
    main()